In [2]:
import json,os
import random
import string
from apis import *
from datetime import datetime, timedelta

with open("ontology.json",'r') as f:
    ontology = json.load(f)
    
day_list = ["sunday","monday","tuesday","wednesday","thursday","friday","saturday"]
stay_list = [str(idx) for idx in range(1,9)]
people_list = [str(idx) for idx in range(1,9)]

In [3]:
[slot for slot in ontology if "taxi" in slot]

['taxi-arriveBy', 'taxi-departure', 'taxi-destination', 'taxi-leaveAt']

In [3]:
with open("../../data.json",'r') as f:
    data_dict = json.load(f)

In [30]:
def goal_preprocess_text(dial):
    current_goal = ".\n".join(dial['goal']['message'])+"."
    current_goal = current_goal.replace("<span class='emphasis'>","'").replace("</span>","'")
    
    current_goal = current_goal.replace("'places to go'","places to go").replace("'place to stay'","place to stay").replace("'place to dine'","place to dine")
    
    print(current_goal)
    # return current_goal

idx_list = []
for data_idx in data_dict:
    if data_dict[data_idx]['goal']['taxi']:
        # goal_preprocess_text(data_dict[data_idx])
        idx_list.append(data_idx)

random_idx = random.sample(idx_list,1)[0]
goal_preprocess_text(data_dict[random_idx])

data_dict[random_idx]['goal']['taxi']

You are traveling to Cambridge and looking forward to try local restaurants.
You are looking for a 'restaurant'. The restaurant should be in the 'east' and should serve 'south indian' food.
If there is no such restaurant, how about one that serves 'chinese' food.
Once you find the 'restaurant' you want to book a table for '3 people' at '17:45' on 'sunday'.
If the booking fails how about '16:45'.
Make sure you get the 'reference number'.
You are also looking for a place to stay. The hotel should 'include free wifi' and should have 'a star of 4'.
The hotel should 'include free parking'.
Make sure you get 'price range' and 'address'.
You also want to book a 'taxi' to commute between the two places.
You want to make sure it arrives the restaurant 'by the booked time'.
Make sure you get 'contact number' and 'car type'.


{'reqt': ['car type', 'phone'], 'book': {'arriveBy': '16:45'}, 'fail_book': {}}

In [8]:
reqt = set()
for data_idx in data_dict:
    if data_dict[data_idx]['goal']['taxi']:
        if "reqt" in data_dict[data_idx]['goal']['taxi']:
            reqt=reqt^set(data_dict[data_idx]['goal']['taxi']['reqt'])
reqt

{'car type', 'phone'}

In [ ]:
def generate_random_phone_number():
    number = '010' + ''.join(str(random.randint(0, 9)) for _ in range(8))
    return number

def taxi_book(leaveAt:str=None,arriveBy:str=None,destination:str=None,departure:str=None):
    ## 모두 required니까 다 들어왔는지 확인
    input_params = {"leaveAt":leaveAt,"arriveBy":arriveBy,"destination":destination,"departure":departure}
    for param in input_params:
        if input_params[param] is None:
            return {"success":False,"result":{"message":f"Input parameter '{param}' is required parameter."}}
    input_params_type = {"leaveAt":str,"arriveBy":int,"destination":str,"departure":str}
    for param in input_params:
        if not isinstance(input_params[param],input_params_type[param]):
            return {"success":False,"result":{"message":f"Type of {param} must be {input_params_type[param]}"}}
        
    ## 모두 type 맞게 입력되었는지 확인
    book_info = {"leaveAt":leaveAt,"arriveBy":arriveBy,"destination":destination,"departure":departure}
    with open("reservation/taxi_book_db.json",'r') as f:
        taxi_book_db = json.load(f)
    if len(taxi_book_db)==0:
        reservation_id = 0
    else:
        reservation_id = taxi_book_db[-1]['reservation_id']+1
    book_info={"reservation_id":reservation_id,"reference_number":generate_random_string()}|book_info
    
    ## car type, phone number 생성기
    taxi_types = ["toyota","skoda","bmw","honda","ford","audi","lexus","volvo","volkswagen","tesla"]
    book_info['car_type'] = random.sample(taxi_types,1)[0]
    book_info['phone'] = generate_random_phone_number()
    taxi_book_db.append(book_info)
    with open("reservation/taxi_book_db.json",'w') as f:
        json.dump(taxi_book_db,f,indent=4)
    return {"success":True,"result":{"message":f"Booking Success. Booked information: {book_info}"}}

In [2]:
def generate_random_string(length=8):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))
    

def train_book(train_schedule_id:int=None,people:int=None):
    ## pointer_load
    with open("current_db_pointer.txt",'r') as f:
        pointer = f.read()
    
    input_params = {"train_schedule_id":train_schedule_id,"people":people}
    ## 모두 required니까 다 들어왔는지 확인
    for param in input_params:
        if input_params[param] is None:
            return {"success":False,"result":{"message":f"Input parameter '{param}' is required parameter."}}
    
    ## input parameter의 type 확인
    input_params_type = {"train_schedule_id":int,"people":int}
    for param in input_params:
        if not isinstance(input_params[param],input_params_type[param]):
            return {"success":False,"result":{"message":f"Type of {param} must be {input_params_type[param]}"}}
        
    ## id가 db에 있는 것인지 확인
    with open(f"/home/jeonghoonshim/multiwoz_env/data/environment/{pointer}/db/train_db.json",'r') as f:
        train_db = json.load(f)
    train_id = {int(entity['train_schedule_id']):entity['train_schedule_id'] for entity in train_db}
    if input_params['train_schedule_id'] not in train_id:
        return {"success":False,"result":{"message":f"Train schedule id '{input_params['train_schedule_id']}' does not exsist. Please check the id properly."}}
    
    ## 모두 type 맞게 입력되었는지 확인
    book_info = {"train_schedule_id":train_schedule_id,"people":people}
    with open("reservation/train_book_db.json",'r') as f:
        train_book_db = json.load(f)
    if len(train_book_db)==0:
        reservation_id = 0
    else:
        reservation_id = train_book_db[-1]['reservation_id']+1
    book_info={"reservation_id":reservation_id,"reference_number":generate_random_string()}|book_info
    
    ## 여기서 no_book에 해당하는지 확인
    if "train_no_book_db.json" in os.listdir(f"/home/jeonghoonshim/multiwoz_env/data/environment/{pointer}/db"):
        with open(f"/home/jeonghoonshim/multiwoz_env/data/environment/{pointer}/db/train_no_book_db.json",'r') as f:
            train_no_book_db = json.load(f)
        
        #####작업필요######
        for train_no_book in train_no_book_db:
            if train_no_book['train_schedule_id'] == book_info['train_schedule_id']: ## 될려나 모르겠다
                cnt=0
                for slot in ["people"]: ## train 예약에서 제약은 사람 수만?
                    if str(book_info[slot]) == train_no_book[slot]:
                        cnt+=1
                if cnt == 1: ## 현재 예약하려는 조건이 no_book의 조건 중 하나랑 완전히 일치할 경우
                    ## 단, 뭐땜에 예약 안됬는지를 안알려줌
                    return {"success":False,"result":{"message":"Reservation not possible under given conditions (Fully occupied)"}}
        #####작업필요######
    
    ## 모든 시련을 통과함
    train_book_db.append(book_info)
    with open("reservation/train_book_db.json",'w') as f:
        json.dump(train_book_db,f,indent=4)
    return {"success":True,"result":{"message":f"Booking Success. Booked information: {book_info}"}}

In [6]:
# python run_api.py "train_book" """{'train_schedule_id':160,'people':7}"""

# train_book(**{'train_schedule_id':160,'people':7})

In [1]:
from apis import *

attraction_retrieve(**{'area':'cambridge','type':'college'})

{'success': True, 'result': []}